In [560]:
import json
import glob
import pandas as pd
from pprint import pprint
import pandas as pd
import warnings



In [119]:
def split_combined_data_dicts(dict_to_split):

    split_data_dict = {}

    # test_fight_stats['sig_str_per_round']
    for key, value_list in dict_to_split.items():
        for i, val in enumerate(value_list):

            ## create new key based on inext
            new_key = f"{key}_{chr(97 + i)}"
            split_data_dict[new_key] = val

    return split_data_dict

def seperate_composite_columns(comp_dict):

    composite_free_dict = {}

    for key, val in comp_dict.items():

        if len(val.split(" of ")) == 2:

            landed_val = val.split(" of ")[0]
            attempted_val = val.split(" of ")[1]
            composite_free_dict['landed_' + key] = landed_val
            composite_free_dict['attempted_' + key] = attempted_val

        else:

            composite_free_dict[key] = val

    return composite_free_dict


def process_percentages(dict):

    for key in dict.keys():

        if 'percentage' in key:

            dict[key] = dict[key].replace("%", "").replace("---", "0")

    return dict

def process_per_round_dict(per_round_dict):

    split_dict = {}

    for round in per_round_dict.keys():

        split_dict[round] = process_percentages(seperate_composite_columns(split_combined_data_dicts(per_round_dict[round])))

    return split_dict

In [ ]:
def process_fight(raw_fight):

    if 'error' in raw_fight:
        error_dict = {}

        return {"ERROR" : "Drop entry"}


    raw_fight['fight_totals_split'] = process_percentages(seperate_composite_columns(split_combined_data_dicts(raw_fight['fight_totals'])))
    raw_fight['totals_per_round_split'] = process_per_round_dict(raw_fight['totals_per_round'])
    raw_fight['sig_str_split'] = process_percentages(seperate_composite_columns(split_combined_data_dicts(raw_fight['sig_str'])))
    raw_fight['sig_str_per_round_split'] = process_per_round_dict(raw_fight['sig_str_per_round'])

    ## only select
    keep_keys = ['fight_name','results','fight_totals_split', 'totals_per_round_split', 'sig_str_split', 'sig_str_per_round_split']

    # Standard comprehension
    keep_dict = {k: raw_fight[k] for k in keep_keys if k in raw_fight}
    return keep_dict

def process_fights_in_event(raw_event):

    processed_event = {}

    for key, val in raw_event.items():

        ## every va
        if key in ['event_details', 'FUTURE EVENT']:

            processed_event[key] = val

        else:
            processed_val = process_fight(val)
            processed_event[key] = processed_val
        

    return processed_event



dict_values([{'date': 'March 29, 2025', 'location': 'Mexico City, Distrito Federal, Mexico'}, {'results': {'method': 'Decision - Unanimous', 'round': '5', 'time': '5:00', 'time format': '5 Rnd (5-5-5-5-5)', 'referee': 'Herb Dean', 'details:': "Mike Bell 46 - 49.Sal D'amato 46 - 49.Chris Lee 46 - 49."}, 'fight_totals': {'fighters': ['Brandon Moreno', 'Steve Erceg'], 'knockdowns': ['0', '0'], 'significant strikes': ['89 of 176', '116 of 279'], 'significant strike percentage': ['50%', '41%'], 'total strike': ['95 of 182', '119 of 282'], 'takedowns': ['1 of 5', '0 of 0'], 'takedown percentage': ['20%', '---'], 'submission attempts': ['0', '0'], 'reversals': ['0', '0'], 'control': ['1:25', '0:00']}, 'totals_per_round': {'Round 1': {'fighters': ['Brandon Moreno', 'Steve Erceg'], 'knockdowns': ['0', '0'], 'significant strikes': ['21 of 40', '25 of 68'], 'significant strike percentage': ['52%', '36%'], 'total strike': ['21 of 40', '25 of 68'], 'takedowns': ['0 of 0', '0 of 0'], 'takedown perce

In [ ]:
test_dir = 'data/test_5_files'
test_output = 'sample_data_5_events.json'

def merge_JsonFiles(data_dir, output_file_name):
    result = {}
    file_paths = glob.glob(data_dir + '/*.json')

    for file in file_paths:
        with open(file, 'r') as infile:
                raw_event = json.load(infile)
                # print(list(raw_event.keys())[0])
                # print(type(raw_event))
                processed_event = {}

                for key, val in raw_event.items():
                     if val == 'FUTURE EVENT':
                          continue

                     processed_event[key] = process_fights_in_event(val)
                # processed_event = process_fights_in_event(raw_event[list(raw_event.keys())[0]])
                
                result = result | processed_event

    with open(output_file_name, 'w') as output_file:
        json.dump(result, output_file, indent=4)

merge_JsonFiles(test_dir, test_output)



In [499]:
with open('/Users/macuser/projects/ufc_prediction/sample_data_5_events.json', 'r') as infile:
    sample_data = json.load(infile)

test_fight = sample_data['0f7210aa8d61af8d']['33e301872e2b5680']
test_fight.keys()

dict_keys(['fight_name', 'results', 'fight_totals_split', 'totals_per_round_split', 'sig_str_split', 'sig_str_per_round_split'])

In [ ]:
def reformat_per_round_dict(per_round_dict):

    reformatted_dict = {}

    for round_num, round_stats_dict in per_round_dict.items():

        for key, val in round_stats_dict.items():
            reformmatted_col_name = round_num + ' ' + key
            reformatted_dict[reformmatted_col_name] = val

    return reformatted_dict


def dataframe_row_from_fight(fight_dict):

    if 'ERROR' in fight_dict:
        return 'ERROR'

    df_name = pd.DataFrame({'matchup':[fight_dict['fight_name']]})
    df_results = pd.DataFrame([fight_dict['results']])
    df_fight_totals = pd.DataFrame([fight_dict['fight_totals_split']])
    df_totals_per_round = pd.DataFrame([reformat_per_round_dict(fight_dict['totals_per_round_split'])])
    df_sig_str = pd.DataFrame([fight_dict['sig_str_split']])
    df_sig_str_per_round = pd.DataFrame([reformat_per_round_dict(fight_dict['sig_str_per_round_split'])])

    df_full = pd.concat([df_name, df_results, df_fight_totals, df_totals_per_round, df_sig_str, df_sig_str_per_round], axis = 1)
    return df_full


In [505]:
df_cols_file = pd.read_csv('data_schema.csv')
# df_cols_file['col_name'].value_counts()

counts = df_cols_file['col_name'].value_counts()
dup_cols = counts[counts > 1].index.tolist()
dup_cols

## for a given fight, loop through the list of known duplicate column names,

def deduplicate_fight_df_row(fight_df_row, dup_cols):

    
    applicable_dup_cols = [col for col in dup_cols if col in fight_df_row.columns.tolist()]
    
    ## First, verify that the duplicate columns have the same value. If a duplicate column has different values, then we have a problem
    for dup_col in applicable_dup_cols:
        
        ## get both duplicate column rows for a given duplicate column name
        dup_df_val = fight_df_row[dup_col]

        if dup_df_val.iloc[0,0] == dup_df_val.iloc[0,1]:

            pass

        else:
            ## if the values are different, raise an error
            raise ValueError("duplicate columns have different values, there is a mismatch in the data")
    
    # Remove one of the duplicated columns
    fight_df_row_dedup = fight_df_row.loc[:,~fight_df_row.columns.duplicated()]

    return fight_df_row_dedup


In [401]:
cols = pd.read_csv('data_schema_dedup_raw.csv')
cols['cols_clean'] = cols.index.str.replace("'","").str.lstrip()
# cols['cols_clean']
# cols_clean = cols['col'].str.replace("'","").str.lstrip()
cols['cols_clean'].to_csv('data_schema_dedup.csv', index=False)

In [570]:
## create the database schema
df_cols_dedup_file = pd.read_csv('data_schema_dedup.csv')
df_cols = df_cols_dedup_file['col_name'].tolist()
df_schema = pd.DataFrame(columns = df_cols)

def dataframe_from_event(event_dict, df_schema, dup_cols_list):

    event_df = df_schema.copy()

    for fight in event_dict:
        if fight != 'event_details':
            df_fight_row = dataframe_row_from_fight(event_dict[fight])

            # if df_fight_row == 'ERROR':

            if  isinstance(df_fight_row, pd.DataFrame):
                df_fight_row_dedup = deduplicate_fight_df_row(df_fight_row, dup_cols_list)
                event_df = pd.concat([event_df, df_fight_row_dedup], ignore_index = True)

            else:
                msg = "no data listed for fight: " + fight
                warnings.warn(msg)
                continue


    event_df['event_date'] = event_dict['event_details']['date']
    event_df['event_location'] = event_dict['event_details']['location']

    return event_df


all_dat = df_schema.copy()
for event in sample_data:

    event_df = dataframe_from_event(sample_data[event], df_schema, dup_cols)

    all_dat = pd.concat([all_dat, 
event_df])

all_dat.head()
print(all_dat.shape)

(59, 309)


In [540]:
## main function -- merge seperated data
seperated_data_dir = 'data/all_events_seperated'
merged_output = '/Users/macuser/projects/ufc_prediction/merged_data/all_events.json'
merge_JsonFiles(seperated_data_dir, merged_output)



In [571]:
## main function -- process data
def process_fight_data(merged_data_file, dup_cols_file, schema_file):

    with open(merged_data_file, 'r') as infile:
        events_data = json.load(infile)

    ## Load in the dup cols list
    dup_cols = pd.read_csv(dup_cols_file)
    dup_col_list = dup_cols['col_name'].tolist()

    ## Load in the df schema
    df_scehma_file = pd.read_csv(schema_file)
    df_cols = df_scehma_file['col_name'].tolist()
    df_schema = pd.DataFrame(columns = df_cols)

    ## initialize a df object for the full dataset
    proccessed_dat = df_schema.copy()

    ## iterate through every event in the data, convert each event to a dataframe and add it to the full data
    for event in events_data:

        # print(event)

        event_df = dataframe_from_event(events_data[event], df_schema, dup_col_list)

        proccessed_dat = pd.concat([proccessed_dat, event_df])

    return proccessed_dat


In [572]:
merged_output_all = '/Users/macuser/projects/ufc_prediction/merged_data/all_events.json'
merged_output_100 = '/Users/macuser/projects/ufc_prediction/merged_data/most_recent_100_events.json'
dup_cols_file = 'duplicate_columns.csv'
schema_file = 'data_schema_dedup.csv'
processed_df_all = process_fight_data(merged_output_all, dup_cols_file, schema_file)
print(processed_df_all.shape)
processed_df_all.head(10)


/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_81790/3593634100.py:22: UserWarning: no data listed for fight: e4fe950846b51bdf
  warnings.warn(msg)
/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_81790/3593634100.py:22: UserWarning: no data listed for fight: 4b334c9727eee450
  warnings.warn(msg)
/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_81790/3593634100.py:22: UserWarning: no data listed for fight: 3badedeb2c5533f4
  warnings.warn(msg)
/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_81790/3593634100.py:22: UserWarning: no data listed for fight: c413b0abc04358c3
  warnings.warn(msg)
/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_81790/3593634100.py:22: UserWarning: no data listed for fight: 8e03db41687d9132
  warnings.warn(msg)
/var/folders/ft/jgrps1md2_30cc3nx_zr573r0000gn/T/ipykernel_81790/3593634100.py:22: UserWarning: no data listed for fight: b297c3e938e1005e
  warnings.warn(msg)
/var/folders/ft/jgrps1md2_30cc3nx_zr573r

(8604, 309)


,matchup,method,round,time,time format,referee,details:,fighters_a,fighters_b,knockdowns_a,...,Round 5 landed_clinch_a,Round 5 attempted_clinch_a,Round 5 landed_clinch_b,Round 5 attempted_clinch_b,Round 5 landed_ground_a,Round 5 attempted_ground_a,Round 5 landed_ground_b,Round 5 attempted_ground_b,event_date,event_location
0,Marlon Vera vs Dominick Cruz,KO/TKO,4,2:17,5 Rnd (5-5-5-5-5),Herb Dean,Kick to Head At Distance,Marlon Vera,Dominick Cruz,3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
1,Nate Landwehr vs David Onama,Decision - Majority,3,5:00,3 Rnd (5-5-5),Mike Beltran,Mike Bell 28 - 28.Junichiro Kamijo 27 - 29.Der...,Nate Landwehr,David Onama,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
2,Yazmin Jauregui vs Iasmin Lucindo,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Jason Herzog,Ron McCarthy 27 - 30.Derek Cleary 28 - 29.Sal ...,Yazmin Jauregui,Iasmin Lucindo,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
3,Devin Clark vs Azamat Murzakanov,KO/TKO,3,1:18,3 Rnd (5-5-5),Frank Trigg,Punch to Body At Distance,Devin Clark,Azamat Murzakanov,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
4,Priscila Cachoeira vs Ariane da Silva,KO/TKO,1,1:05,3 Rnd (5-5-5),Herb Dean,Punches to Head At Distance,Priscila Cachoeira,Ariane da Silva,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
5,Bruno Silva vs Gerald Meerschaert,Submission,3,1:39,3 Rnd (5-5-5),Mike Beltran,Guillotine Choke On Ground,Bruno Silva,Gerald Meerschaert,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
6,Angela Hill vs Loopy Godinez,Decision - Unanimous,3,5:00,3 Rnd (5-5-5),Jason Herzog,Derek Cleary 28 - 29.Junichiro Kamijo 28 - 29....,Angela Hill,Loopy Godinez,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
7,Martin Buday vs Lukasz Brzeski,Decision - Split,3,5:00,3 Rnd (5-5-5),Frank Trigg,Sal D'amato 28 - 29.Mike Bell 29 - 28.Eliot Ke...,Martin Buday,Lukasz Brzeski,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
8,Cynthia Calvillo vs Nina Nunes,Decision - Split,3,5:00,3 Rnd (5-5-5),Herb Dean,Derek Cleary 29 - 28.Junichiro Kamijo 28 - 29....,Cynthia Calvillo,Nina Nunes,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"
9,Gabriel Benitez vs Charlie Ontiveros,KO/TKO,1,3:33,3 Rnd (5-5-5),Mike Beltran,Punches to Head From Mount,Gabriel Benitez,Charlie Ontiveros,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"August 13, 2022","San Diego, California, USA"


In [ ]:
processed_df_all.to_csv('processed_fight_dat.csv')

768